# Giai đoạn 1: Thu thập Dữ liệu Giá Lịch sử VN100 (2022 - 2026)

**Môn học:** Học máy (Machine Learning) - HK7  
**Đề tài:** Phân nhóm cổ phiếu theo mức độ đồng biến động giá phục vụ đa dạng hóa danh mục đầu tư  

---

### Mục tiêu của Notebook này:
1. Lấy danh sách toàn bộ các mã cổ phiếu thuộc rổ **VN100** và bảng tra cứu nhóm ngành (**ICB Sector**).
2. Thu thập dữ liệu giá lịch sử (OHLCV) từng ngày từ **01/01/2022 đến 23/10/2026** cho từng mã cổ phiếu.
3. Lưu trữ dữ liệu thô vào thư mục `data/raw/{symbol}.csv` và metadata vào `data/metadata/vn100_symbols.csv`.
4. Kiểm tra sơ bộ tính toàn vẹn (số lượng bản ghi, khoảng thời gian) của dữ liệu thô vừa thu thập.

In [ ]:
# Khai báo các thư viện cần thiết
import sys
from pathlib import Path
import pandas as pd

# Thiết lập đường dẫn thư mục gốc dự án
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import get_vn100_symbols, fetch_stock_history, save_symbol_data

print("Đã import thành công các module cần thiết!")

### 1. Lấy danh sách rổ cổ phiếu VN100 & Phân ngành ICB
Rổ chỉ số VN100 bao gồm 30 mã VN30 (vốn hóa lớn nhất) và 70 mã VNMidCap (vốn hóa trung bình có thanh khoản cao nhất sàn HOSE). Việc lưu lại phân ngành ICB là bắt buộc để phục vụ nhiệm vụ đối chiếu phân cụm học máy với ngành nghề thực tế.

In [ ]:
# Lấy danh sách mã VN100
symbols, df_meta = get_vn100_symbols()
print(f"Tổng số mã cổ phiếu được nhận diện: {len(symbols)}")
print("10 mã đầu tiên:", symbols[:10])

# Lưu metadata vào data/metadata/vn100_symbols.csv
meta_path = PROJECT_ROOT / "data" / "metadata" / "vn100_symbols.csv"
meta_path.parent.mkdir(parents=True, exist_ok=True)
df_meta.to_csv(meta_path, index=False, encoding="utf-8-sig")
print(f"Đã lưu metadata tại: {meta_path}")

### 2. Thu thập dữ liệu giá OHLCV của từng mã (2022 - 2026)
Chúng ta sẽ lặp qua danh sách mã cổ phiếu và tải dữ liệu từ ngày `2022-01-01` đến `2026-10-23`.  
Mỗi mã sẽ được lưu vào một file CSV riêng tại `data/raw/{symbol}.csv` để tránh mất dữ liệu nếu mạng bị ngắt quãng giữa chừng.

In [ ]:
import time
from tqdm import tqdm

start_date = "2022-01-01"
end_date = "2026-10-23"
raw_dir = PROJECT_ROOT / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

success_list = []
fail_list = []

print(f"Bắt đầu cào dữ liệu {len(symbols)} mã từ {start_date} đến {end_date}...")
for sym in tqdm(symbols, desc="Thu thập giá VN100"):
    target_csv = raw_dir / f"{sym}.csv"
    if target_csv.exists():
        success_list.append(sym)
        continue
        
    df_price = fetch_stock_history(sym, start_date=start_date, end_date=end_date, source="AUTO")
    if df_price is not None and not df_price.empty:
        save_symbol_data(df_price, sym, raw_dir)
        success_list.append(sym)
    else:
        fail_list.append(sym)
    time.sleep(0.5)  # Tránh rate limit

print(f"\nHoàn tất: Thành công {len(success_list)}/{len(symbols)} mã.")
if fail_list:
    print("Các mã gặp sự cố:", fail_list)

### 3. Kiểm tra sơ bộ dữ liệu thô vừa thu thập
Đọc mẫu 1 file dữ liệu (ví dụ mã đầu ngành HPG hoặc VCB) để quan sát các trường thông tin.

In [ ]:
sample_sym = "HPG"
sample_file = raw_dir / f"{sample_sym}.csv"
if sample_file.exists():
    df_sample = pd.read_csv(sample_file)
    print(f"=== Thông tin mẫu dữ liệu {sample_sym} ===")
    print(f"Số dòng: {len(df_sample)}")
    print("Các cột:", list(df_sample.columns))
    display(df_sample.head())
else:
    print(f"Chưa tìm thấy file {sample_file}")

### Kết luận & Định hướng bước tiếp theo (Phase 2):
Dữ liệu thô của các mã VN100 đã được lưu trữ an toàn. Ở bước tiếp theo (Notebook `02`), chúng ta sẽ:
1. Lọc và xử lý căn chỉnh ngày giao dịch chung cho tất cả các mã.
2. Kiểm tra tỷ lệ khuyết dữ liệu (missing values) và quyết định danh sách mã đạt chuẩn.
3. Tính chuỗi lợi suất logarit $r_t = \ln(P_t / P_{t-1})$ và xây dựng ma trận tương quan.